# Limpeza técnica do corpus (coluna de texto) — Greve USP 2023

Este notebook lê um arquivo **.xlsx** com colunas como `ID`, `Data`, `Ator`, `Grupo do Ator` e `Transcrição Integral da Publicação`, limpa **apenas a coluna de texto** (removendo ruído técnico e de formatação, sem alterar conteúdo discursivo) e devolve um **novo .xlsx**, preservando as demais colunas intactas.

**O que este notebook NÃO faz:** correção gramatical, lematização, stemming, remoção de stopwords, resumo, substituição de sinônimos, remoção de negações ou de termos políticos.

Execute as células em ordem, de cima para baixo.

## Célula 1 — Importação das bibliotecas
Todas as bibliotecas usadas já vêm instaladas no Google Colab por padrão.

In [ ]:
import re
import unicodedata
import io
from collections import Counter

import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from google.colab import files

print("Bibliotecas carregadas com sucesso.")

## Célula 2 — Upload do arquivo .xlsx
Selecione o arquivo original. Ele **nunca** será sobrescrito — todo o processamento gera um arquivo novo.

In [ ]:
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Por favor, envie exatamente UM arquivo .xlsx por vez.")

ORIGINAL_FILENAME = list(uploaded.keys())[0]
if not ORIGINAL_FILENAME.lower().endswith(('.xlsx', '.xlsm')):
    raise ValueError(f"O arquivo '{ORIGINAL_FILENAME}' não parece ser um .xlsx. Verifique o arquivo enviado.")

print(f"Arquivo carregado: {ORIGINAL_FILENAME}")
print(f"Tamanho: {len(uploaded[ORIGINAL_FILENAME])} bytes")

## Célula 3 — Detecção da aba e das colunas

O código tenta identificar automaticamente:
- a **aba (sheet)** que contém o texto integral (procura por uma aba cujo nome contenha "texto", ou usa a única aba existente, ou a primeira aba);
- a **coluna de texto** a ser limpa (procura por um cabeçalho contendo "transcri" — ex: "Transcrição Integral da Publicação");
- a **coluna de ID**, para o relatório de auditoria (procura por cabeçalho "id").

**Se a detecção automática errar**, edite manualmente as variáveis `SHEET_NAME_OVERRIDE` e `TEXT_COLUMN_OVERRIDE` / `ID_COLUMN_OVERRIDE` logo abaixo e rode a célula novamente — não é necessário mexer no restante do notebook.

In [ ]:
# --- Se a detecção automática falhar, preencha manualmente aqui (ou deixe None) ---
SHEET_NAME_OVERRIDE = None      # ex: "Texto Completo"
TEXT_COLUMN_OVERRIDE = None     # ex: "Transcrição Integral da Publicação"
ID_COLUMN_OVERRIDE = None       # ex: "ID"
# ------------------------------------------------------------------------------

xls = pd.ExcelFile(io.BytesIO(uploaded[ORIGINAL_FILENAME]))
sheet_names = xls.sheet_names
print("Abas encontradas no arquivo:", sheet_names)

if SHEET_NAME_OVERRIDE:
    SHEET_NAME = SHEET_NAME_OVERRIDE
elif len(sheet_names) == 1:
    SHEET_NAME = sheet_names[0]
else:
    candidatas = [s for s in sheet_names if 'texto' in s.lower()]
    SHEET_NAME = candidatas[0] if candidatas else sheet_names[0]

print(f"\nAba selecionada: '{SHEET_NAME}'")
if not SHEET_NAME_OVERRIDE and len(sheet_names) > 1:
    print("(detecção automática — confirme se é a aba correta; caso não seja, "
          "defina SHEET_NAME_OVERRIDE acima e rode esta célula novamente)")

df_original = pd.read_excel(io.BytesIO(uploaded[ORIGINAL_FILENAME]), sheet_name=SHEET_NAME)
print(f"\nColunas encontradas: {list(df_original.columns)}")
print(f"Número de linhas (unidades de análise): {len(df_original)}")

if TEXT_COLUMN_OVERRIDE:
    TEXT_COLUMN = TEXT_COLUMN_OVERRIDE
else:
    candidatas_texto = [c for c in df_original.columns if 'transcri' in str(c).lower()]
    if not candidatas_texto:
        raise ValueError(
            "Não foi possível detectar automaticamente a coluna de texto. "
            "Defina TEXT_COLUMN_OVERRIDE com o nome exato da coluna e rode esta célula novamente. "
            f"Colunas disponíveis: {list(df_original.columns)}"
        )
    TEXT_COLUMN = candidatas_texto[0]

if ID_COLUMN_OVERRIDE:
    ID_COLUMN = ID_COLUMN_OVERRIDE
else:
    candidatas_id = [c for c in df_original.columns if str(c).strip().lower() == 'id']
    ID_COLUMN = candidatas_id[0] if candidatas_id else None

print(f"\nColuna de texto a ser limpa: '{TEXT_COLUMN}'")
print(f"Coluna de ID (para o relatório): '{ID_COLUMN}'")

df_original.head(3)

## Célula 4 — Funções de limpeza

Cada transformação é isolada em sua própria função e **contabiliza** o que remove, para permitir o relatório de auditoria. Nenhuma função altera a ordem das palavras, corrige gramática ou remove stopwords.

In [ ]:
# ---------------------------------------------------------------------------
# Contadores globais de auditoria (acumulados ao longo de todo o corpus)
# ---------------------------------------------------------------------------
stats = Counter()
mentions_found = []   # lista de (ID, menção) para revisão manual
hashtags_found = []   # lista de (ID, hashtag original, resultado)

URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)

# Faixas Unicode típicas de emojis/pictogramas
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"  # símbolos, pictogramas, emoticons, transporte, etc.
    "\U00002600-\U000027BF"  # símbolos diversos e dingbats
    "\U0001F1E6-\U0001F1FF"  # bandeiras (pares de letras regionais)
    "\U00002190-\U000021FF"  # setas (parte de alguns emojis compostos)
    "\U00002B00-\U00002BFF"  # setas e símbolos diversos adicionais
    "\U0000FE0F"              # seletor de variação (variation selector)
    "\U0000200D"              # zero width joiner (usado em emojis compostos)
    "]+",
    flags=re.UNICODE
)

# Marcador técnico "audio:" / "áudio:" / "legenda:" — removido onde quer que apareça no
# texto (não só no início), pois uma mesma célula pode conter mais de uma transcrição
# (ex: vários trechos de áudio separados por "|"). Usa \b para não confundir com
# palavras como "audiovisual".
MARKER_PATTERN = re.compile(r'(?i)\b(audio|áudio|legenda)\s*:\s*')

MENTION_PATTERN = re.compile(r'@(\w[\w.]*)')
HASHTAG_PATTERN = re.compile(r'#(\w+)')

# Pontuação removida sem deixar espaço (não corre risco de grudar palavras,
# pois normalmente já estão cercadas de espaço ou de outra palavra)
PUNCT_NO_SPACE = '.,!?:;()[]{}'
# Aspas e apóstrofos: removidos sem deixar espaço (envolvem a palavra, não a separam de outra)
QUOTE_CHARS = '"\'\u201c\u201d\u2018\u2019'
# Caracteres que, ao serem removidos, PODEM grudar palavras — substituídos por espaço
# (inclui o separador '|', usado no corpus para marcar diferentes trechos/transcrições
# dentro da mesma célula, sem valor analítico próprio)
SPACE_SUBSTITUTE_CHARS = '-–—*|'

MOJIBAKE_CHAR = '\ufffd'  # o caractere �


def remove_urls(text):
    urls = URL_PATTERN.findall(text)
    stats['urls_removidas'] += len(urls)
    return URL_PATTERN.sub(' ', text)


def remove_emojis(text):
    emojis = EMOJI_PATTERN.findall(text)
    # findall com + pode agrupar sequências; contar caracteres individuais é mais fiel
    n_chars = sum(len(e) for e in emojis)
    stats['emojis_removidos'] += n_chars if n_chars else len(emojis)
    return EMOJI_PATTERN.sub(' ', text)


def remove_markers(text):
    matches = MARKER_PATTERN.findall(text)
    for m in matches:
        if m.lower() in ('audio', 'áudio'):
            stats['marcadores_audio_removidos'] += 1
        else:
            stats['marcadores_legenda_removidos'] += 1
    return MARKER_PATTERN.sub('', text)


def extract_and_strip_mentions(text, row_id):
    def _replace(match):
        mention = match.group(1)
        mentions_found.append((row_id, '@' + mention))
        stats['mencoes_encontradas'] += 1
        return mention
    return MENTION_PATTERN.sub(_replace, text)


def split_camel_case(word):
    """Insere espaço entre uma letra minúscula/dígito e uma maiúscula seguinte
    (ex: 'ContrataçõesJa' -> 'Contrações Ja'), e entre uma sequência de dígitos e
    uma letra. Não tenta separar blocos de maiúsculas consecutivas (ex: 'USP'),
    para evitar segmentação arriscada."""
    # letra minúscula/acentuada seguida de maiúscula -> insere espaço
    step1 = re.sub(r'(?<=[a-zà-ÿ0-9])(?=[A-ZÀ-Ý])', ' ', word)
    # dígito seguido de letra, ou letra seguida de dígito -> insere espaço
    step2 = re.sub(r'(?<=[0-9])(?=[A-Za-zÀ-ÿ])|(?<=[A-Za-zÀ-ÿ])(?=[0-9])', ' ', step1)
    return step2


def extract_and_process_hashtags(text, row_id):
    def _replace(match):
        tag = match.group(1)
        stats['hashtags_encontradas'] += 1
        # só tenta segmentar se houver mistura de maiúsculas/minúsculas (sinal de CamelCase)
        has_lower = any(c.islower() for c in tag)
        has_upper = any(c.isupper() for c in tag)
        if has_lower and has_upper:
            processed = split_camel_case(tag)
            stats['hashtags_processadas_camelcase'] += 1
        else:
            # bloco todo maiúsculo, todo minúsculo, ou com números apenas:
            # preserva sem tentar segmentar (evita inventar separação insegura)
            processed = tag
        hashtags_found.append((row_id, '#' + tag, processed))
        return processed
    return HASHTAG_PATTERN.sub(_replace, text)


def remove_quotes(text):
    n = sum(text.count(c) for c in QUOTE_CHARS)
    stats['aspas_apostrofos_removidos'] += n
    for c in QUOTE_CHARS:
        text = text.replace(c, '')
    return text


def replace_space_substitute_chars(text):
    n = sum(text.count(c) for c in SPACE_SUBSTITUTE_CHARS)
    stats['hifens_asteriscos_removidos'] += n
    for c in SPACE_SUBSTITUTE_CHARS:
        text = text.replace(c, ' ')
    return text


def remove_basic_punctuation(text):
    n = sum(text.count(c) for c in PUNCT_NO_SPACE)
    stats['pontuacao_removida'] += n
    for c in PUNCT_NO_SPACE:
        text = text.replace(c, ' ' if c in ':;' else '')
    return text
    # obs: ':' e ';' viram espaço por segurança (podem aparecer sem espaço ao redor,
    # ex: 'às17h30'); os demais sinais quase sempre já têm espaço ao lado.


def normalize_whitespace(text):
    # o texto final deve ficar "corrido" (uma única linha, sem quebras de parágrafo):
    # substitui qualquer sequência de espaços/tabs/quebras de linha por um único espaço.
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def count_mojibake(text):
    n = text.count(MOJIBAKE_CHAR)
    stats['caracteres_mojibake_encontrados'] += n
    return n


def clean_text(raw_text, row_id):
    """Aplica a sequência completa de limpeza técnica a um texto de uma unidade."""
    text = str(raw_text)
    count_mojibake(text)
    text = remove_markers(text)
    text = remove_urls(text)
    text = remove_emojis(text)
    text = extract_and_strip_mentions(text, row_id)
    text = extract_and_process_hashtags(text, row_id)
    text = remove_quotes(text)
    text = replace_space_substitute_chars(text)
    text = remove_basic_punctuation(text)
    text = text.lower()
    text = normalize_whitespace(text)
    return text

print("Funções de limpeza definidas.")

## Célula 5 — Aplicação da limpeza

A limpeza é aplicada **apenas** à coluna de texto identificada na Célula 3. As demais colunas (`ID`, `Data`, `Ator`, `Grupo do Ator` etc.) são copiadas sem qualquer alteração. Células vazias (sem transcrição) permanecem vazias — nenhum texto é inventado.

In [ ]:
df_limpo = df_original.copy()

NEW_COLUMN_NAME = "Texto_Limpo_IRaMuTeQ"

textos_limpos = []
n_unidades_vazias = 0

for _, row in df_original.iterrows():
    row_id = row[ID_COLUMN] if ID_COLUMN else row.name
    valor_original = row[TEXT_COLUMN]

    if pd.isna(valor_original) or str(valor_original).strip() == '':
        n_unidades_vazias += 1
        textos_limpos.append(None)  # mantém vazio — não inventa texto
        continue

    textos_limpos.append(clean_text(valor_original, row_id))

# insere a nova coluna logo após a coluna de texto original
insert_pos = df_limpo.columns.get_loc(TEXT_COLUMN) + 1
df_limpo.insert(insert_pos, NEW_COLUMN_NAME, textos_limpos)

stats['unidades_vazias'] = n_unidades_vazias
stats['total_unidades'] = len(df_original)
stats['linhas_antes'] = len(df_original)
stats['linhas_depois'] = len(df_limpo)

print(f"Limpeza aplicada a {len(df_original) - n_unidades_vazias} unidades.")
print(f"{n_unidades_vazias} unidades sem texto (permaneceram vazias, conforme esperado).")

df_limpo[[c for c in [ID_COLUMN, TEXT_COLUMN, NEW_COLUMN_NAME] if c]].head(3)

## Célula 6 — Geração das estatísticas de auditoria
Consolida os contadores em tabelas (DataFrames) para exportação.

In [ ]:
resumo_rows = [
    ("Arquivo original", ORIGINAL_FILENAME),
    ("Aba processada", SHEET_NAME),
    ("Coluna de texto processada", TEXT_COLUMN),
    ("Total de unidades (linhas)", stats['total_unidades']),
    ("Unidades vazias (sem transcrição)", stats['unidades_vazias']),
    ("Linhas antes da limpeza", stats['linhas_antes']),
    ("Linhas depois da limpeza", stats['linhas_depois']),
    ("URLs removidas", stats['urls_removidas']),
    ("Hashtags encontradas", stats['hashtags_encontradas']),
    ("Hashtags processadas (CamelCase seguro)", stats['hashtags_processadas_camelcase']),
    ("Menções (@) encontradas", stats['mencoes_encontradas']),
    ("Emojis/símbolos removidos", stats['emojis_removidos']),
    ("Marcadores 'audio:' removidos", stats['marcadores_audio_removidos']),
    ("Marcadores 'legenda:' removidos", stats['marcadores_legenda_removidos']),
    ("Caracteres problemáticos de codificação (�) encontrados", stats['caracteres_mojibake_encontrados']),
    ("Pontuação básica removida (. , ! ? : ;)", stats['pontuacao_removida']),
    ("Aspas/apóstrofos removidos", stats['aspas_apostrofos_removidos']),
    ("Hífens/asteriscos removidos (substituídos por espaço)", stats['hifens_asteriscos_removidos']),
]
df_relatorio = pd.DataFrame(resumo_rows, columns=["Métrica", "Valor"])

df_mencoes = pd.DataFrame(mentions_found, columns=["ID", "Menção encontrada"]) if mentions_found else \
    pd.DataFrame(columns=["ID", "Menção encontrada"])

df_hashtags = pd.DataFrame(hashtags_found, columns=["ID", "Hashtag original", "Resultado processado"]) if hashtags_found else \
    pd.DataFrame(columns=["ID", "Hashtag original", "Resultado processado"])

print(df_relatorio.to_string(index=False))

## Célula 7 — Verificação de segurança

Confere rapidamente que acentos/caracteres portugueses foram preservados e mostra uma amostra antes/depois para conferência manual.

In [ ]:
acentos_teste = 'çãõáàâéêíóôúü'
amostra_com_acento = df_limpo[NEW_COLUMN_NAME].dropna().astype(str)
tem_acento = amostra_com_acento.str.contains('|'.join(list(acentos_teste)), regex=True).any()
print(f"Caracteres acentuados presentes no texto limpo? {'SIM' if tem_acento else 'NÃO — verificar!'}")

print(f"\nCaracteres problemáticos (�) remanescentes no texto limpo: "
      f"{df_limpo[NEW_COLUMN_NAME].dropna().astype(str).str.count('\ufffd').sum()}")

print("\n--- Amostra antes / depois (primeira unidade não vazia) ---")
amostra = df_original[df_original[TEXT_COLUMN].notna()].iloc[0]
idx = amostra.name
print("ANTES:\n", df_original.loc[idx, TEXT_COLUMN][:400])
print("\nDEPOIS:\n", df_limpo.loc[idx, NEW_COLUMN_NAME][:400])

## Célula 8 — Salvamento do arquivo limpo (.xlsx)

Gera um novo arquivo com:
- a aba original **acrescida** da coluna `Texto_Limpo_IRaMuTeQ` (o arquivo/aba original não é sobrescrito — isso é um arquivo novo);
- uma aba `Relatorio_Auditoria` com as estatísticas;
- uma aba `Mencoes_Para_Revisao` com as menções encontradas, para revisão manual;
- uma aba `Hashtags_Processadas` mostrando cada hashtag original e o resultado da segmentação.

In [ ]:
OUTPUT_FILENAME = "corpus_greve_USP_2023_limpo.xlsx"

with pd.ExcelWriter(OUTPUT_FILENAME, engine="openpyxl") as writer:
    df_limpo.to_excel(writer, sheet_name=SHEET_NAME[:31], index=False)
    df_relatorio.to_excel(writer, sheet_name="Relatorio_Auditoria", index=False)
    df_mencoes.to_excel(writer, sheet_name="Mencoes_Para_Revisao", index=False)
    df_hashtags.to_excel(writer, sheet_name="Hashtags_Processadas", index=False)

print(f"Arquivo salvo como: {OUTPUT_FILENAME}")
print("Abas geradas:", [SHEET_NAME[:31], "Relatorio_Auditoria", "Mencoes_Para_Revisao", "Hashtags_Processadas"])

## Célula 9 — Download do arquivo

In [ ]:
files.download(OUTPUT_FILENAME)